# MongoDB Capstone: University Registration Database

You are the database owner for a university registrar. In seven short steps you will design, load, query, protect, speed up, search and finally check a registration database, using one skill from each earlier lab.

Each step has one or two **TODO** markers and a **check cell** that tells you when you are done. Do the steps in order. Stuck? Compare with the solution notebook.

In [ ]:
!pip install -qU "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi

### Step 0 — Connect

In [ ]:
import os
import time
import random
import datetime as dt
import certifi
from dotenv import load_dotenv
import pymongo
from pymongo.operations import SearchIndexModel

load_dotenv("../../.env")
client = pymongo.MongoClient(os.environ["MONGODB_URI"], tlsCAFile=certifi.where())

db = client["registrar_db"]
students, courses, enrollments = db["students"], db["courses"], db["enrollments"]

for coll in (students, courses, enrollments):
    coll.drop()                       # start every run from empty collections

print("Connected. registrar_db is empty and ready.")

### Step 1 — Design and Seed the Database

*Skills from: Labs 1, 4, 4B*

Create the three collections the registrar needs and load the starting data.

**Schema decisions (already made for you, notice why):**

| Collection | Holds | Pattern used |
|---|---|---|
| `students` | one document per student | plain document |
| `courses` | one per course, with a `seats_available` counter | **computed pattern** (a stored counter, never recounted) |
| `enrollments` | one per student per course, with a copy of `course_title` | **referencing** by `student_id` and `course_id` plus an **extended reference** for the title |

**Your tasks**
1. Insert `STUDENTS` and `COURSES`.
2. Create a **unique** index on `enrollments` over `(student_id, course_id)` named `uniq_student_course`, so nobody can enroll twice in the same course.

In [ ]:
PROGRAMS = ["Computer Science", "Mathematics", "Physics", "Biology", "English"]

STUDENTS = [
    {"student_id": f"S{n:03d}", "name": f"Student {n:02d}", "program": PROGRAMS[n % 5],
     "year": 1 + n % 4, "status": "active" if n % 7 else "inactive"}
    for n in range(1, 21)
]

COURSES = [
    {"course_id": "CS101", "title": "Introduction to Programming", "program": "Computer Science",
     "description": "Learn variables, loops and functions by writing small Python programs.",
     "seats_total": 3, "seats_available": 3},
    {"course_id": "CS201", "title": "Data Structures", "program": "Computer Science",
     "description": "Lists, trees and graphs, and how to choose the right structure for a problem.",
     "seats_total": 40, "seats_available": 40},
    {"course_id": "MA101", "title": "Calculus I", "program": "Mathematics",
     "description": "Limits, derivatives and integrals with applications to science.",
     "seats_total": 40, "seats_available": 40},
    {"course_id": "PH101", "title": "Classical Mechanics", "program": "Physics",
     "description": "Motion, forces and energy explained through experiments.",
     "seats_total": 40, "seats_available": 40},
    {"course_id": "BI101", "title": "Cell Biology", "program": "Biology",
     "description": "How cells work, divide and communicate.",
     "seats_total": 40, "seats_available": 40},
    {"course_id": "EN101", "title": "Academic Writing", "program": "English",
     "description": "Plan, draft and revise clear essays and reports.",
     "seats_total": 40, "seats_available": 40},
]

# TODO 1: insert STUDENTS into students and COURSES into courses


# TODO 2: create a UNIQUE index on enrollments over (student_id, course_id), named "uniq_student_course"


# ---- check ----
assert students.count_documents({}) == 20, "expected 20 students"
assert courses.count_documents({}) == 6, "expected 6 courses"
assert enrollments.index_information().get("uniq_student_course", {}).get("unique"), "unique index missing"
print("Step 1 check passed")

### Step 2 — Query and Maintain the Data

*Skills from: Labs 1, 2*

Everyday registrar work: look things up and keep records correct.

**Your tasks**
1. Find the **active Computer Science students**, sorted by name. Store the list in `cs_students`.
2. Student `S014` is marked inactive by mistake. Set their `status` to `"active"`.
3. Add a new course `CS301` ("Databases") using an **upsert**, and run the upsert **twice** to prove it never creates a duplicate.

*Hints:* `find(filter).sort(...)`, `update_one(..., {"$set": ...})`, and `upsert=True` with `$setOnInsert` for fields that should only be written when the document is first created.

In [ ]:
# TODO 1: cs_students = list of active Computer Science students, sorted by name
cs_students = []


# TODO 2: set S014's status to "active"


# TODO 3: upsert course CS301 ("Databases", program "Computer Science", 40 seats). Run it twice.
new_course = {"course_id": "CS301", "title": "Databases", "program": "Computer Science",
              "description": "Design, query and tune document databases.", "seats_total": 40}
for _ in range(2):
    pass   # replace with your upsert


# ---- check ----
assert [s["student_id"] for s in cs_students] == ["S005", "S010", "S015", "S020"], "wrong CS students"
assert students.find_one({"student_id": "S014"})["status"] == "active", "S014 not reactivated"
assert courses.count_documents({"course_id": "CS301"}) == 1, "upsert created a duplicate or nothing"
print("Step 2 check passed")

### Step 3 — Enroll Students Safely with a Transaction

*Skills from: Labs 2, 5*

Enrolling a student changes two collections at once: a seat is taken in `courses` and a record is added to `enrollments`. Both must happen, or neither.

**Your tasks** — complete the `enroll` function:
1. Confirm the student exists and is **active**, otherwise raise `ValueError`.
2. Take one seat: update the course only if `seats_available > 0`, decrementing the counter, and raise `ValueError` if no seat was available.
3. Insert the enrollment record, copying the course `title` into `course_title` (the extended reference).

The notebook then enrolls three students into `CS101` (only 3 seats), tries a 4th (must fail), and tries an inactive student (must fail).

*Hints:* `find_one_and_update(filter, update, session=session)` returns `None` when the filter matches nothing. Every operation inside the transaction needs `session=session`.

In [ ]:
def enroll(session, student_id, course_id):
    # TODO a: confirm the student exists and is active, else raise ValueError("student not eligible")

    # TODO b: take one seat (only if seats_available > 0); raise ValueError("no seats available") if none

    # TODO c: insert the enrollment with course_title copied from the course, status "active", enrolled_at now
    pass


def run_enroll(student_id, course_id):
    with client.start_session() as session:
        session.with_transaction(lambda s: enroll(s, student_id, course_id))


for sid in ("S001", "S002", "S003"):
    run_enroll(sid, "CS101")
print("Three students enrolled in CS101")

for sid, why in (("S004", "course is full"), ("S007", "student is inactive")):
    try:
        run_enroll(sid, "CS101")
        print("ERROR: this should have failed:", sid)
    except ValueError as err:
        print(f"Rejected {sid} ({why}): {err}")


# ---- check ----
cs101 = courses.find_one({"course_id": "CS101"})
assert cs101["seats_available"] == 0, "CS101 should have 0 seats left"
assert enrollments.count_documents({"course_id": "CS101"}) == 3, "expected exactly 3 enrollments"
assert enrollments.find_one({"student_id": "S004"}) is None, "S004 must not be enrolled"
assert enrollments.find_one({"student_id": "S001"})["course_title"] == "Introduction to Programming"
print("Step 3 check passed")

### Step 4 — Report on the Data with an Aggregation

*Skills from: Lab 3*

The dean wants a report: for every course, how many **active** enrollments does it have and what is the **average grade**, highest average first.

**Your task:** write the aggregation pipeline `report_pipeline` using `$match`, `$group` (with `$sum` and `$avg`) and `$sort`. Then run it.

*Hints:* match `status == "active"` first; group by `$course_id`; name the output fields `enrolled` and `avg_grade`.

In [ ]:
# Given: load 30,000 synthetic enrollments (5,000 visiting students x 6 courses) so the data is big enough to matter
random.seed(11)
START = dt.datetime(2025, 1, 1)
titles = {c["course_id"]: c["title"] for c in COURSES}

batch, total = [], 0
for b in range(1, 5001):
    for cid in titles:
        batch.append({
            "student_id": f"B{b:05d}", "course_id": cid, "course_title": titles[cid],
            "status": random.choices(["active", "completed", "withdrawn"], weights=[70, 20, 10])[0],
            "enrolled_at": START + dt.timedelta(minutes=random.randint(0, 60 * 24 * 365)),
            "grade": random.randint(40, 100),
        })
        if len(batch) == 10_000:
            enrollments.insert_many(batch); total += len(batch); batch = []
if batch:
    enrollments.insert_many(batch); total += len(batch)

print("Synthetic enrollments loaded:", total)
print("Total enrollment documents:", enrollments.count_documents({}))

# TODO: report_pipeline = [ $match active, $group by course_id (enrolled, avg_grade), $sort by avg_grade descending ]
report_pipeline = []

assert report_pipeline, "write report_pipeline first"
report = list(enrollments.aggregate(report_pipeline))
for row in report:
    print(f"{row['_id']:<6} enrolled={row['enrolled']:>5}  avg_grade={row['avg_grade']:.1f}")


# ---- check ----
assert len(report) == 6, "expected one row per seeded course"
assert sum(r["enrolled"] for r in report) == enrollments.count_documents({"status": "active"}), "counts do not add up"
avgs = [r["avg_grade"] for r in report]
assert avgs == sorted(avgs, reverse=True), "report must be sorted by avg_grade descending"
print("Step 4 check passed")

### Step 5 — Make the Dashboard Query Fast

*Skills from: Labs 3, 3B*

The registrar's dashboard shows the **10 newest active enrollments for a course**. With 30,000 documents and no matching index, that query reads everything.

**Your task:** create a compound index that follows the **ESR rule** (Equality fields, then Sort field) for this query, then re-check the plan.

The two equality fields are `course_id` and `status`; the sort field is `enrolled_at` (newest first, so descending). Name the index `esr_course_status_date`.

*Success looks like:* `COLLSCAN` and a `SORT` stage before; `IXSCAN` and no `SORT` stage after, with about 10 documents examined instead of 30,000.

In [ ]:
def collect_stages(node, found=None):
    found = [] if found is None else found
    if isinstance(node, dict):
        if "stage" in node:
            found.append(node["stage"])
        for value in node.values():
            collect_stages(value, found)
    elif isinstance(node, list):
        for value in node:
            collect_stages(value, found)
    return found

def plan_of(cursor):
    report = cursor.explain()
    stages = list(dict.fromkeys(collect_stages(report["queryPlanner"]["winningPlan"])))
    stats = report["executionStats"]
    return {"stages": stages, "docs": stats["totalDocsExamined"], "returned": stats["nReturned"]}

def latest_active(course_id):
    """The 10 newest active enrollments for one course (a dashboard query)."""
    return enrollments.find({"course_id": course_id, "status": "active"}).sort("enrolled_at", -1).limit(10)

before = plan_of(latest_active("CS201"))
print("Before:", before)

# TODO: create the ESR compound index for latest_active() named "esr_course_status_date"


after = plan_of(latest_active("CS201"))
print("After: ", after)


# ---- check ----
assert any("COLLSCAN" in s for s in before["stages"]), "baseline should be a collection scan"
assert any("IXSCAN" in s for s in after["stages"]), "query should now use an index"
assert not any("SORT" in s for s in after["stages"]), "no in-memory SORT should remain"
assert after["docs"] <= 20, "should examine about 10 documents"
print("Step 5 check passed")

### Step 6 — Let Students Search the Course Catalog

*Skills from: Lab 4C*

Students type course names with typos. Exact `find()` queries cannot cope, but **Atlas Search** can.

The notebook creates an Atlas Search index on `courses` (title and description) and waits for it to build.

**Your task:** write `search_courses(text)`: a `$search` pipeline using the `text` operator on `title` and `description` with **fuzzy matching (one typo allowed)**, limited to 3 results, returning only `course_id` and `title`.

*Test it with `"programing"` (one `m` missing): the top result must be `CS101`.*

*Hints:* `$search` must be the first stage; fuzzy is `{"maxEdits": 1}`. This step needs Atlas (the free M0 tier is fine).

In [ ]:
SEARCH_INDEX = "course_search"

if list(courses.list_search_indexes(SEARCH_INDEX)):
    courses.drop_search_index(SEARCH_INDEX)
    time.sleep(10)

courses.create_search_index(model=SearchIndexModel(
    definition={"mappings": {"dynamic": False, "fields": {
        "title": {"type": "string"}, "description": {"type": "string"}}}},
    name=SEARCH_INDEX, type="search"))

print("Building search index (20-60 seconds)...")
while True:
    found = list(courses.list_search_indexes(SEARCH_INDEX))
    if found and found[0].get("queryable"):
        break
    time.sleep(5)
print("Search index ready")


def search_courses(text):
    # TODO: return a pipeline: $search (text on title + description, fuzzy maxEdits 1), $limit 3, $project course_id and title
    return []


results = list(courses.aggregate(search_courses("programing")))
print(results)


# ---- check ----
assert results and results[0]["course_id"] == "CS101", "CS101 should be the top result for 'programing'"
print("Step 6 check passed")

### Step 7 — Check the Deployment and Clean Up

*Skills from: Lab 6*

The last job of a database owner: confirm the system is deployed safely, then leave no mess behind.

**Your tasks**
1. Run the `hello` command and store the replica set name in `set_name` and the number of members in `member_count`.
2. Set `tls_enforced` to `True` if the connection string uses the `mongodb+srv` scheme (which requires TLS).
3. Use `connectionStatus` to list your current user's roles in `roles`.
4. Clean up: drop the search index, then the whole `registrar_db` database.

*Hints:* `db.command("hello")` returns `setName` and `hosts`; `db.command("connectionStatus")["authInfo"]["authenticatedUserRoles"]` lists roles.

In [ ]:
# TODO 1: hello = db.command("hello"); set_name and member_count from it
set_name, member_count = None, 0

# TODO 2: tls_enforced = True when the URI scheme is mongodb+srv
tls_enforced = False

# TODO 3: roles = list of role names for the current user (from connectionStatus)
roles = []

print("Replica set:", set_name, "| members:", member_count, "| TLS enforced:", tls_enforced)
print("Current roles:", roles)


# ---- check ----
assert set_name and member_count >= 1, "read the replica set from hello"
assert tls_enforced is True, "mongodb+srv connections always use TLS"
assert len(roles) >= 1, "list the current user's roles"

# TODO 4: clean up. Drop the "course_search" search index on courses, then drop the whole registrar_db database


print("Step 7 check passed")

### Done

If every check printed *passed*, your registration database is complete. Finish the reflection questions in the assignment file.